## 3. Cloner le projet

`/kaggle/input` est en lecture seule : on travaille dans
`/kaggle/working`, inscriptible et conserve comme sortie du
notebook.


## 1. Verifier le GPU


In [ ]:
!nvidia-smi


## 2. Installer les dependances

Kaggle fournit deja torch avec CUDA : on n'installe qu'ultralytics.


In [ ]:
!pip install -q ultralytics
import torch, ultralytics
print('torch', torch.__version__, '| CUDA :', torch.cuda.is_available())
print('GPU  :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'aucun')


# Le projet est clone depuis GitHub : plus besoin de creer un
# Dataset Kaggle ni de televerser une archive.
# Verifiez seulement que 'Internet' est sur 'On' dans le panneau
# de droite, sinon le clone et pip echouent.
import os
!git clone -q https://github.com/othmanedhilou/MODELE.git /kaggle/working/MODELE
os.chdir('/kaggle/working/MODELE')
print('Repertoire de travail :', os.getcwd())
!ls


In [ ]:
# Le projet est clone depuis GitHub : plus besoin de creer un
# Dataset Kaggle ni de televerser une archive.
# Verifiez seulement que 'Internet' est sur 'On' dans le panneau
# de droite, sinon le clone et pip echouent.
import os
!git clone -q https://github.com/othmanedhilou/MODELE.git /kaggle/working/MODELE
os.chdir('/kaggle/working/MODELE')
print('Repertoire de travail :', os.getcwd())
!ls


## 4. Generer les datasets synthetiques (si vous n'avez pas de donnees)

Deux des trois modeles s'entrainent sans aucune image de l'usine.
Sautez cette etape si vos donnees annotees sont deja dans l'archive.

Les modeles obtenus ne sont pas deployables tels quels : ils valident
la chaine et servent de point de depart a affiner sur 50 a 300 images
reelles. Voir `docs/sans_donnees.md`.


In [ ]:
!python scripts/generer_dataset_convoyeur.py --nombre 600
!python scripts/generer_dataset_eclairage.py --nombre 600


Le modele vehicules n'a pas de generateur synthetique : une voiture
dessinee ne se transfere pas au reel. Importez un dataset public :


In [ ]:
# !python -m src.prepare.importer_dataset --source telechargements/vehicules --inspecter
# !python -m src.prepare.importer_dataset --source telechargements/vehicules \
#        --modele vehicules --correspondance configs/correspondance_vehicules.yaml


## 5. Controler le dataset

Si cette cellule annonce 0 image, inutile de continuer :
l'entrainement echouera.


In [ ]:
!python -m src.prepare.check_dataset --modele convoyeur


## 6. Entrainer

| Modele | Resolution | Epochs | Duree approximative (P100) |
|--------|-----------|--------|----------------------------|
| Vehicules | 640 | 120 | 1 h 15 |
| Eclairage | 960 | 150 | 1 h 45 |
| Convoyeur (segmentation) | 1024 | 200 | 2 h 30 |

En cas de `CUDA out of memory`, reduisez `--batch` (4, puis 2).


In [ ]:
!python -m src.train.train --modele convoyeur


In [ ]:
# !python -m src.train.train --modele eclairage


In [ ]:
# !python -m src.train.train --modele vehicules


## 7. Evaluer sur le lot de test

Seul chiffre presentable comme performance reelle. L'evaluation
s'inscrit au registre MLOps avec l'empreinte du dataset.


In [ ]:
!python -m src.train.evaluer --modele convoyeur --exporter onnx
!python -m src.mlops.registre --lister


### Courbes pour le rapport


In [ ]:
from IPython.display import Image, display
import glob
for chemin in sorted(glob.glob('runs/convoyeur/train/*.png')):
    print(chemin)
    display(Image(chemin, width=700))


## 8. Rassembler les poids a recuperer

Tout ce qui se trouve dans `/kaggle/working` a la fin est
telechargeable depuis l'onglet **Output**. On rassemble les fichiers
utiles pour eviter de fouiller l'arborescence.


In [ ]:
import shutil, os
os.makedirs('/kaggle/working/resultats', exist_ok=True)
for modele in ('convoyeur', 'eclairage', 'vehicules'):
    source = f'/kaggle/working/MODELE/runs/{modele}/train/weights/best.pt'
    if os.path.exists(source):
        shutil.copy(source, f'/kaggle/working/resultats/{modele}_best.pt')
        print('copie :', modele)
shutil.copy('/kaggle/working/MODELE/models/registre.json',
            '/kaggle/working/resultats/registre.json')
!ls -lh /kaggle/working/resultats


---
## Apres l'entrainement

Sur le poste local, placez chaque `best.pt` dans
`runs/<modele>/train/weights/`, puis promouvez-le :

```bash
python -m src.mlops.registre --promouvoir convoyeur --version v1
python -m src.pipeline.run_stream --camera cam_convoyeur_01
```

Le pipeline ne charge que les poids promus : un modele reste inactif
tant que vous ne l'avez pas explicitement mis en production.
